# 02 - Action: 方案确定与方法选择

> 从"不知道怎么做"到"逐步确定方案"

---

## 一、方法探索：从失败中学习

### 1.1 失败路径1：简单回归

**想法**：用线性回归，价格→订单量，看系数正负。

**问题**：

In [ ]:
# TODO: 演示简单回归的问题

import pandas as pd
import numpy as np
import statsmodels.api as sm

# 加载数据
df = pd.read_csv('../data/raw/rider_pricing_clean.csv')

# 简单回归：价格 → 订单量
X = sm.add_constant(df['price'])
y = df['order_volume']

model_naive = sm.OLS(y, X).fit()
print(model_naive.summary())

print("\n" + "="*60)
print("问题分析：")
print("1. 系数为负：价格越高，订单量越少")
print("2. 但这是'相关'，不是'因果'")
print("3. 雨天本来订单就少，不是因为价格高")
print("4. 忽略了选择偏差：价格不是随机分配的")
print("="*60)

print("TODO: 思考为什么这个结论可能是错的")

### 1.2 失败路径2：AB实验幻想

**想法**：随机给不同区域定价，看效果。

**问题**：

In [ ]:
# TODO: 分析为什么AB实验不可行

print("""
AB实验不可行的原因：

1. 用户体验问题
   - 同一区域，今天5元，明天8元 → 用户困惑、投诉
   - 损害品牌信任

2. 商业逻辑问题
   - 雨天必须加价（骑手安全补贴）
   - 不可能"随机"不加价
   - 运营团队不会同意

3. 伦理问题
   - 随机加价可能损害部分用户利益
   - 平台责任与商业利益的平衡

结论：观察性因果推断是唯一可行路径
""")

print("TODO: 列出AB实验不可行的具体原因")

### 1.3 失败路径3：PSM维度灾难

**想法**：用倾向得分匹配（PSM），匹配"加价"和"不加价"的区域。

**问题**：

In [ ]:
# TODO: 分析PSM的局限性

print("""
PSM的局限性：

1. 高维匹配困难
   - 区域×时段组合：10区域 × 24时段 = 240种组合
   - 加上天气、节假日等，维度爆炸
   - 匹配质量差，很多样本找不到匹配对象

2. 连续处理变量问题
   - PSM适用于二值处理（加价/不加价）
   - 但价格是连续的（3元~15元）
   - 需要离散化，损失信息

3. 模型依赖
   - 倾向得分模型如果误设，匹配结果偏差
   - 需要检验平衡性，操作复杂

结论：需要更现代的方法
""")

print("TODO: 计算区域×时段的组合数，理解维度灾难")

## 二、正确路径：因果推断框架

### 2.1 为什么需要因果推断？

```
相关 ≠ 因果

观察到的：雨天价格高，订单少
错误结论：加价导致订单减少

真相：雨天本来订单就少（用户不想出门）
       价格只是"伴随"，不是"原因"

因果推断的目标：分离"伴随"与"因果"
```

### 2.2 方法选择决策树

In [ ]:
# TODO: 绘制方法选择决策树

print("""
方法选择决策树：

Q1: 能随机实验吗？
    ├─ 能 → 随机实验（黄金标准）
    └─ 不能 → 继续

Q2: 处理变量是连续还是二值？
    ├─ 二值 → PSM / 双重差分 / 工具变量
    └─ 连续 → 继续

Q3: 需要异质性效应吗？
    ├─ 不需要 → 简单回归调整
    └─ 需要 → 继续

Q4: 协变量维度高吗？
    ├─ 不高 → 传统方法（回归调整）
    └─ 高 → DML + Meta-Learners + 因果森林

我们的情况：
- 不能随机实验 ✗
- 连续处理变量（价格）✗
- 需要异质性效应 ✗
- 高维协变量 ✗

结论：DML + Meta-Learners + 因果森林
""")

print("TODO: 根据决策树，确认我们的方法选择")

## 三、核心方法详解

### 3.1 DML（双重机器学习）

**为什么用DML？**

```
传统方法的问题：
- 高维控制：区域×时段×天气×...，维度灾难
- 模型误设：如果函数形式错了，估计有偏

DML的解决：
1. 用ML估计nuisance函数（倾向得分、结果模型）
2. 样本分割：避免过拟合
3. 正交化：减少模型误设的影响
```

**核心公式**：

```
Y = θ·D + g(X) + ε
D = m(X) + η

其中：
- Y: 结果（订单量）
- D: 处理（价格）
- X: 协变量（天气、区域、时段...）
- g(X): 结果模型（用ML估计）
- m(X): 倾向得分（用ML估计）
- θ: 我们关心的因果效应
```

### 3.2 Meta-Learners（S/T/X/R-Learner）

**为什么对比不同Learner？**

```
不同场景下，最优策略不同：

S-Learner: 简单，但可能忽略处理效应
T-Learner: 分别建模，但方差大
X-Learner: 利用倾向得分，适合处理组少的情况
R-Learner: 残差化，对模型误设更稳健

没有"最好"的Learner，只有"最适合"的Learner
```

In [ ]:
# TODO: 对比S/T/X/R-Learner的原理

print("""
S/T/X/R-Learner对比：

S-Learner（Single Learner）:
  原理：用一个模型，处理变量作为特征之一
  公式：μ(x,d) = E[Y|X=x, D=d]
  优点：简单，不容易过拟合
  缺点：如果处理效应弱，模型可能忽略D
  适用：处理效应强，样本量大

T-Learner（Two Learner）:
  原理：分别对处理组和对照组建模
  公式：τ(x) = μ₁(x) - μ₀(x)
  优点：灵活，能捕捉复杂效应
  缺点：方差大，尤其处理组/对照组不平衡时
  适用：处理组和对照组样本量相当

X-Learner（Cross Learner）:
  原理：利用倾向得分，对处理组少的场景优化
  公式：组合处理组和对照组的伪结果
  优点：处理组少时表现好
  缺点：依赖倾向得分估计
  适用：处理组远少于对照组

R-Learner（Residual Learner）:
  原理：先残差化，再估计效应
  公式：τ(x) = argmin E[(Ỹ - τ(X)·D̃)²]
  优点：对nuisance模型误设更稳健
  缺点：需要好的nuisance估计
  适用：高维场景，模型可能误设
""")

print("TODO: 根据数据特点，预判哪个Learner可能最优")

### 3.3 因果森林

**为什么用因果森林？**

```
传统方法的问题：
- 需要预设交互项（区域×天气×时段）
- 维度高时，组合爆炸
- 不知道哪些交互重要

因果森林的解决：
1. 自动发现异质性子群（数据驱动）
2. 不需要预设交互项
3. 提供置信区间（知道估计有多可靠）
```

### 3.4 双重稳健估计

**为什么需要双重稳健？**

```
现实：模型可能误设
- 倾向得分模型错了？→ 估计有偏
- 结果模型错了？→ 估计有偏

双重稳健的保证：
- 只要两个模型中有一个是对的，估计就一致
- "双重保险"

类比：
- 飞机有两个引擎，一个坏了还能飞
- 双重稳健：一个模型错了，另一个能救
```

## 四、方法选择总结

In [ ]:
# TODO: 总结方法选择矩阵

import pandas as pd

method_comparison = pd.DataFrame({
    '方法': ['简单回归', 'AB实验', 'PSM', 'DML', 'Meta-Learners', '因果森林', '双重稳健'],
    '能否用': ['能，但有偏', '不能', '能，但困难', '能', '能', '能', '能'],
    '处理选择偏差': ['否', '是', '部分', '是', '是', '是', '是'],
    '处理异质性': ['否', '是', '否', '是', '是', '是', '是'],
    '处理高维': ['否', '是', '否', '是', '是', '是', '是'],
    '置信区间': ['有', '有', '无', '有', '有', '有', '有'],
    '复杂度': ['低', '高', '中', '高', '高', '高', '高']
})

print(method_comparison.to_string(index=False))

print("\n" + "="*60)
print("最终方案：")
print("1. DML：处理高维协变量 + 选择偏差")
print("2. S/T/X/R-Learner：对比不同策略")
print("3. 因果森林：自动发现异质性子群")
print("4. 双重稳健：模型误设保护")
print("5. 交叉验证：选择最优Learner")
print("="*60)

print("TODO: 确认方法选择，准备进入数据探索")

## 五、工具选择理由

### 为什么用A不用B？

| 问题 | 为什么不用B | 为什么用A |
|------|------------|----------|
| 高维协变量 | 传统回归：维度灾难 | DML：ML处理高维 |
| 异质性 | 回归交互项：预设、组合爆炸 | 因果森林：自动发现 |
| 模型误设 | 单模型：错了就全错 | 双重稳健：双重保险 |
| 选择偏差 | 简单回归：混淆因果 | DML：正交化分离 |
| 策略选择 | 拍脑袋：主观 | 交叉验证：数据驱动 |

### 核心原则

```
每个工具选择都有明确理由
不是"因为高级所以用"
而是"因为需要所以用"
```

## 六、下一步

1. **数据探索**：了解数据分布、质量、潜在问题
2. **快速验证**：用简单方法验证假设方向
3. **详细建模**：用选定的方法精确估计

---

> **核心洞察**：方法选择不是越复杂越好，而是越"适合"越好。从失败中学习，才能找到正确的路。